In [1]:
#SMOTE BASED BALANCING AUGMENTATION:

In [16]:
import os
import cv2 
import gc
import numpy as np 
from tqdm import tqdm
from PIL import Image  
import shutil
import random 
from glob import glob
import tensorflow as tf
from collections import defaultdict 
from sklearn.neighbors import NearestNeighbors 
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [5]:
base_path = r"D:\final year project\all datasets\Smote_approach\Second Set"
preprocessed_path1 = r"D:\final year project\all datasets\Smote_approach\resized_set"
preprocessed_path2 = r"D:\final year project\all datasets\Smote_approach\denoised_set"
preprocessed_path3 = r"D:\final year project\all datasets\Smote_approach\enhanced_set"
preprocessed_path4 = r"D:\final year project\all datasets\Smote_approach\balanced_set"
os.makedirs(preprocessed_path1, exist_ok=True)
os.makedirs(preprocessed_path2, exist_ok=True)
os.makedirs(preprocessed_path3, exist_ok=True)
os.makedirs(preprocessed_path4, exist_ok=True)

In [6]:
# RESIZE AUGMENTED IMAGES

TARGET_SIZE = (256, 256)

def resize_dataset(input_dir, output_dir):
    for class_name in ["400x Normal Oral Cavity Histopathological Images", "400x OSCC Histopathological Images"]:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)
        input_path = os.path.join(input_dir, class_name)
        output_path = os.path.join(output_dir, class_name)
        
        img_files = [f for f in os.listdir(input_path) if f.lower().endswith('.jpg')]
        
        for img_file in tqdm(img_files, desc=f"Resizing {class_name}"):
            try:
                img = cv2.imread(os.path.join(input_path, img_file))
                if img is not None:
                    resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                    cv2.imwrite(os.path.join(output_path, img_file), resized)
            except Exception:
                continue

print("Resizing images...")
resize_dataset(base_path, preprocessed_path1)
print(f"\nResizing complete. Output saved to: {preprocessed_path1}")

Resizing images...


Resizing 400x Normal Oral Cavity Histopathological Images:   0%|          | 0/201 [00:00<?, ?it/s]

Resizing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:27<00:00,  7.20it/s]
Resizing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [01:11<00:00,  6.95it/s]


Resizing complete. Output saved to: D:\final year project\all datasets\Smote_approach\resized_set


In [7]:
# Apply median filter to color image
def median_filter_color(img, kernel_size=3):
    # Ensure kernel size is odd and >= 3
    kernel_size = max(3, kernel_size | 1)
    return cv2.medianBlur(img, kernel_size)

# Denoise function using median filter
def denoise_dataset(input_dir, output_dir, kernel_size=3):
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all class folders
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    
    for class_name in tqdm(class_folders, desc="Processing classes"):
        class_input_dir = os.path.join(input_dir, class_name)
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Get all images in class folder
        image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg'))]
        
        for img_file in tqdm(image_files, desc=f"Processing {class_name}", leave=False):
            img_path = os.path.join(class_input_dir, img_file)
            
            # Load image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}")
                continue
            
            # Apply median filter
            filtered_img = median_filter_color(img, kernel_size=kernel_size)
            
            # Save result
            output_path = os.path.join(class_output_dir, img_file)
            cv2.imwrite(output_path, filtered_img)

# Example usage
if __name__ == "__main__":
    denoise_dataset(preprocessed_path1, preprocessed_path2, kernel_size=3)

Processing classes: 100%|██████████| 2/2 [00:35<00:00, 17.71s/it]


In [8]:
#CLAHE

# Create output directory structure
class_dirs = [d for d in os.listdir(preprocessed_path2) if os.path.isdir(os.path.join(preprocessed_path2, d))]
for class_dir in class_dirs:
    os.makedirs(os.path.join(preprocessed_path3, class_dir), exist_ok=True)

# Apply CLAHE using OpenCV
def apply_clahe_to_image(image_np):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.7, tileGridSize=(8,8))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# Process all images
for class_dir in class_dirs:
    input_class_path = os.path.join(preprocessed_path2, class_dir)
    output_class_path = os.path.join(preprocessed_path3, class_dir)

    image_paths = glob(os.path.join(input_class_path, "*.jpg"))

    for image_path in tqdm(image_paths, desc=f"Processing {class_dir}"):
 
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.uint8)

        image_np = image.numpy()
        
        # Apply CLAHE
        clahe_image = apply_clahe_to_image(image_np)

        # Save output
        image_name = os.path.basename(image_path)
        output_path = os.path.join(output_class_path, image_name)
        cv2.imwrite(output_path, cv2.cvtColor(clahe_image, cv2.COLOR_RGB2BGR)) 

Processing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:09<00:00, 20.45it/s]
Processing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:18<00:00, 26.49it/s]


In [12]:
# Set the seed for reproducibility
random.seed(42)

# Paths
SOURCE_DIR = preprocessed_path3
DEST_DIR = preprocessed_path4

# Splits
SPLITS = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
}

# Create split folders
for split in SPLITS:
    for class_name in os.listdir(SOURCE_DIR):
        class_path = os.path.join(SOURCE_DIR, class_name)
        if os.path.isdir(class_path):
            split_dir = os.path.join(DEST_DIR, split, class_name)
            os.makedirs(split_dir, exist_ok=True)

# Process each class
for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg"))]
    random.shuffle(images)

    total = len(images)
    train_end = int(SPLITS["train"] * total)
    val_end = train_end + int(SPLITS["val"] * total)

    split_data = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    # Copy files with progress bar
    for split, split_images in split_data.items():
        print(f"\nCopying {split} data for class '{class_name}':")
        for img_name in tqdm(split_images, desc=f"{class_name} - {split}", unit="img"):
            src_path = os.path.join(class_path, img_name)
            dest_path = os.path.join(DEST_DIR, split, class_name, img_name)
            shutil.copy2(src_path, dest_path)

print("\n✅ Dataset split completed successfully.")


Copying train data for class '400x Normal Oral Cavity Histopathological Images':


400x Normal Oral Cavity Histopathological Images - train: 100%|██████████| 140/140 [00:01<00:00, 89.75img/s] 



Copying val data for class '400x Normal Oral Cavity Histopathological Images':


400x Normal Oral Cavity Histopathological Images - val: 100%|██████████| 30/30 [00:00<00:00, 93.49img/s]



Copying test data for class '400x Normal Oral Cavity Histopathological Images':


...


Copying val data for class '400x OSCC Histopathological Images':


400x OSCC Histopathological Images - val: 100%|██████████| 74/74 [00:01<00:00, 41.18img/s]



Copying test data for class '400x OSCC Histopathological Images':


400x OSCC Histopathological Images - test: 100%|██████████| 75/75 [00:01<00:00, 47.12img/s]


✅ Dataset split completed successfully.


In [14]:
class ImageSMOTE:
    def __init__(self, input_folder, output_folder, desired_samples_per_class=None, k_neighbors=5, target_size=(256, 256)):
        self.input_folder = input_folder
        self.output_folder = output_folder
        self.desired_samples = desired_samples_per_class
        self.k_neighbors = k_neighbors
        self.target_size = target_size
        self.image_extensions = ('.jpg',)

        os.makedirs(self.output_folder, exist_ok=True)

    def load_images(self):
        class_folders = [d for d in os.listdir(self.input_folder)
                         if os.path.isdir(os.path.join(self.input_folder, d))]

        self.image_paths = defaultdict(list)
        self.image_data = defaultdict(list)
        self.class_names = []

        print("Loading and resizing images>>>")
        for class_name in tqdm(class_folders, desc="Classes"):
            class_path = os.path.join(self.input_folder, class_name)
            image_files = [f for f in os.listdir(class_path)
                           if f.lower().endswith(self.image_extensions)]

            self.class_names.append(class_name)
            for img_file in tqdm(image_files, desc=f"Loading {class_name} images", leave=False):
                img_path = os.path.join(class_path, img_file)
                self.image_paths[class_name].append(img_path)

                img = cv2.imread(img_path)
                if img is None:
                    print(f"Image not found: {img_path}")
                    continue
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                self.image_data[class_name].append(img)

        print("Image loading complete. Summary:")
        for cls in self.class_names:
            print(f"{cls}: {len(self.image_data[cls])} images")

        return self.image_data

    def preprocess_images(self, image_data):
        processed_data = {}

        print("Preprocessing images>>>")
        for class_name, images in tqdm(image_data.items(), desc="Preprocessing Classes"):
            batch_size = 50
            processed_vectors = []

            for i in tqdm(range(0, len(images), batch_size), desc=f"Preprocessing {class_name}", leave=False):
                batch = images[i:i + batch_size]
                vectors = [img.astype(np.float32) / 255.0 for img in batch]
                vectors = [v.flatten() for v in vectors]
                processed_vectors.extend(vectors)

                del batch, vectors
                gc.collect()

            processed_data[class_name] = np.array(processed_vectors, dtype=np.float32)

        print("Preprocessing complete. Summary:")
        for cls in self.class_names:
            print(f"{cls}: {len(processed_data[cls])} vectors")

        return processed_data

    def apply_smote(self, processed_data):
        print(f"Applying SMOTE. Target samples per class: {self.desired_samples}")
        augmented_data = {}

        for class_name in tqdm(self.class_names, desc="SMOTE Classes"):
            current_data = processed_data[class_name]
            current_count = current_data.shape[0]
            target_count = self.desired_samples if self.desired_samples else current_count

            # Save original images first
            class_output_path = os.path.join(self.output_folder, class_name)
            os.makedirs(class_output_path, exist_ok=True)
            original_images = self.image_data[class_name]
            for i, img in enumerate(original_images):
                img_path = os.path.join(class_output_path, f"{class_name}_orig_{i}.jpg")
                Image.fromarray(img).save(img_path)

            if current_count >= target_count:
                indices = np.random.choice(current_count, target_count, replace=False)
                augmented_data[class_name] = current_data[indices]
                continue

            n_neighbors = min(self.k_neighbors, current_count - 1)
            n_synthetic = target_count - current_count
            original_shape = (*self.target_size, 3)
            start_index = len(original_images)

            synthetic_samples = self._generate_synthetic_samples_mem(
                current_data,
                n_synthetic,
                n_neighbors=n_neighbors,
                batch_size=50,
                class_name=class_name,
                original_shape=original_shape,
                save_dir=class_output_path,
                start_index=start_index
            )

            augmented_data[class_name] = np.vstack([current_data, synthetic_samples])
            del synthetic_samples
            gc.collect()

        print("SMOTE complete.")
        return augmented_data

    def _generate_synthetic_samples_mem(self, samples, n_synthetic, n_neighbors=5, batch_size=50,
                                        class_name="", original_shape=(256, 256, 3),
                                        save_dir="", start_index=0):
        synthetic_samples = []
        generated = 0
        pbar = tqdm(total=n_synthetic, desc=f"Generating {class_name} synthetic images")

        knn = NearestNeighbors(n_neighbors=n_neighbors)
        knn.fit(samples)

        while generated < n_synthetic:
            current_batch_size = min(batch_size, n_synthetic - generated)
            batch = []

            for i in range(current_batch_size):
                idx = np.random.randint(0, samples.shape[0])
                sample = samples[idx]
                _, indices = knn.kneighbors([sample])
                neighbor_idx = np.random.choice(indices[0][1:])  # avoid self
                neighbor = samples[neighbor_idx]
                alpha = np.random.random()
                synthetic = sample + alpha * (neighbor - sample)
                batch.append(synthetic)

                # Save image immediately
                img = (synthetic.reshape(original_shape) * 255).clip(0, 255).astype(np.uint8)
                img_path = os.path.join(save_dir, f"{class_name}_aug_{start_index + generated + i}.jpg")
                Image.fromarray(img).save(img_path)

            synthetic_samples.extend(batch)
            generated += current_batch_size
            pbar.update(current_batch_size)

            del batch
            gc.collect()

        pbar.close()
        return np.array(synthetic_samples, dtype=np.float32)

    def run(self):
        try:
            image_data = self.load_images()

            # Dynamically set desired_samples to majority class count
            max_count = max(len(imgs) for imgs in image_data.values())
            self.desired_samples = max_count
            print(f"Setting desired samples per class to the majority class count: {self.desired_samples}")

            processed_data = self.preprocess_images(image_data)
            del image_data
            gc.collect()

            self.apply_smote(processed_data)
            del processed_data
            gc.collect()

            print("SMOTE augmentation and saving completed successfully!")

        except Exception as e:
            print(f"Error during SMOTE augmentation: {str(e)}")
            raise


# Usage
if __name__ == "__main__":  

    org= r"D:\final year project\all datasets\Smote_approach\balanced_set\train"
    dest= r"D:\final year project\all datasets\Smote_approach\balanced_set\train_balanced"
    os.makedirs(dest, exist_ok=True)

    smote = ImageSMOTE(
        input_folder=org,
        output_folder=dest,
        desired_samples_per_class=None,  # Auto-set to majority class
        k_neighbors=5,
        target_size=(256, 256)
    )
    smote.run()


Loading and resizing images>>>


Classes: 100%|██████████| 2/2 [00:04<00:00,  2.22s/it]


Image loading complete. Summary:
400x Normal Oral Cavity Histopathological Images: 140 images
400x OSCC Histopathological Images: 346 images
Setting desired samples per class to the majority class count: 346
Preprocessing images>>>


Preprocessing Classes: 100%|██████████| 2/2 [00:08<00:00,  4.15s/it]


Preprocessing complete. Summary:
400x Normal Oral Cavity Histopathological Images: 140 vectors
400x OSCC Histopathological Images: 346 vectors
Applying SMOTE. Target samples per class: 346


Generating 400x Normal Oral Cavity Histopathological Images synthetic images: 100%|██████████| 206/206 [07:09<00:00,  2.09s/it]
SMOTE Classes: 100%|██████████| 2/2 [07:20<00:00, 220.10s/it]

SMOTE complete.
SMOTE augmentation and saving completed successfully!


In [18]:
# ----------------------- CONFIGURATION -----------------------
import time
# Dataset paths
INPUT_DIR = org
OUTPUT_DIR = r"D:\final year project\all datasets\Smote_approach\balanced_set\train_augmented"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output class folders
classes = [
    "400x Normal Oral Cavity Histopathological Images",
    "400x OSCC Histopathological Images"
]

for cls in classes:
    os.makedirs(os.path.join(OUTPUT_DIR, cls), exist_ok=True)

# Target count per class
TARGET_COUNT = 5000
BATCH_SIZE = 1

# ----------------------- AUGMENTATION SETUP -----------------------

# rotation, width and height shift, horizontal and vertical flip, contrast, brightness

datagen = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True
)

def random_contrast_brightness(image):
    alpha = np.random.uniform(0.7, 1.3)
    beta = np.random.uniform(-7, 7)
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)


# ----------------------- AUGMENTATION FUNCTION -----------------------

def augment_class(class_name, target_count):
    input_path = os.path.join(INPUT_DIR, class_name)
    output_path = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(output_path, exist_ok=True)

    original_images = [
        os.path.join(input_path, f)
        for f in os.listdir(input_path)
        if f.lower().endswith('.jpg')
    ]
    
    existing_augmented = [
        f for f in os.listdir(output_path)
        if f.lower().endswith('.jpg')
    ]

    current_count = len(existing_augmented)
    remaining = target_count - current_count

    if current_count == 0:
        print(f"[INFO] Copying original images for {class_name}")
        for orig_img_path in original_images:
            try:
                img = Image.open(orig_img_path).convert('RGB')
                base_name = os.path.basename(orig_img_path)
                save_path = os.path.join(output_path, f"orig_{base_name}")
                img.save(save_path)
            except Exception as e:
                print(f"[ERROR] Failed to copy image {orig_img_path}: {e}")
        current_count = len(os.listdir(output_path))
        remaining = target_count - current_count

    if remaining <= 0:
        print(f"[INFO] {class_name}: Already has {current_count} images.")
        return

    print(f"\n[INFO] Augmenting '{class_name}' from {current_count} to {target_count} images")
    img_index = 0
    pbar = tqdm(total=remaining, desc=f"Augmenting {class_name}")

    while remaining > 0:
        img_path = original_images[img_index % len(original_images)]

        try:
            with Image.open(img_path).convert('RGB') as img:
                img_array = np.expand_dims(np.array(img), axis=0).astype(np.float32)
                num_augmentations = min(remaining, np.random.randint(4, 7))  # 4 to 6 augmentations

                for i, batch in enumerate(datagen.flow(img_array, batch_size=1)):
                    if i >= num_augmentations:
                        break

                    augmented_img = batch[0]
                    augmented_img = np.uint8(augmented_img)
                    final_img = random_contrast_brightness(augmented_img)

                    filename = f"aug_{os.path.splitext(os.path.basename(img_path))[0]}_{i}_{int(time.time())}.jpg"
                    save_path = os.path.join(output_path, filename)
                    Image.fromarray(final_img).save(save_path)

                    remaining -= 1
                    pbar.update(1)

                    if remaining <= 0:
                        break

        except Exception as e:
            print(f"[ERROR] Failed to process {img_path}: {e}")

        img_index += 1

    pbar.close()
    print(f"[DONE] {class_name}: Total images = {len(os.listdir(output_path))}")

# ----------------------- RUN -----------------------

for cls in classes:
    augment_class(cls, TARGET_COUNT)

# ----------------------- STATS -----------------------

print(f"\n[SUCCESS] Augmentation complete. Output saved to: {OUTPUT_DIR}")
for cls in classes:
    count = len(os.listdir(os.path.join(OUTPUT_DIR, cls)))
    print(f"- {cls}: {count} images")



[INFO] Augmenting '400x Normal Oral Cavity Histopathological Images' from 346 to 5000 images


Augmenting 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 4654/4654 [02:07<00:00, 36.50it/s]


[DONE] 400x Normal Oral Cavity Histopathological Images: Total images = 5000
[INFO] Copying original images for 400x OSCC Histopathological Images

[INFO] Augmenting '400x OSCC Histopathological Images' from 346 to 5000 images


Augmenting 400x OSCC Histopathological Images: 100%|██████████| 4654/4654 [01:51<00:00, 41.92it/s]


[DONE] 400x OSCC Histopathological Images: Total images = 5000

[SUCCESS] Augmentation complete. Output saved to: D:\final year project\all datasets\Smote_approach\balanced_set\train_augmented
- 400x Normal Oral Cavity Histopathological Images: 5000 images
- 400x OSCC Histopathological Images: 5000 images
